# Mangrove condition and carbon exposure screening: Abu Dhabi (UAE) and Nabq (Egypt)

**Theme:** Ecosystem Health, Biodiversity & Blue Carbon (Arab Youth Space Hackathon: 813 Challenge)

**Question.** Which mangrove patches should reserve managers inspect first, and how much carbon sits in them?

**What this notebook does (all open data, run top to bottom):**
1. Builds cloud-screened Sentinel-2 L2A seasonal composites (Jan-Mar) for a baseline year (2019) and a recent year (2025).
2. Corrects the Sentinel-2 processing-baseline offset (+1000 DN shift introduced in January 2022) before comparing years.
3. Restricts the analysis to mangrove pixels (ESA WorldCover class 95) and to a "core" mask that drops edge pixels.
4. Uses Nabq (Egypt), a small and visually stable stand, as a **reference site** to estimate natural variability, and derives the decline threshold from it instead of guessing.
5. Converts area into a carbon **range** using published arid-zone densities. The result is an exposure screening, not a measured carbon loss.

**Important limits (details in the README):** no field validation; NDVI change is an indicator, not proof of stress; one season and two years only; carbon densities come from Qatar and the Red Sea, not from these sites.

In [ ]:
%pip install -q "pystac-client>=0.8,<0.9" planetary-computer stackstac rasterio scipy matplotlib pandas numpy xarray

In [ ]:
import json, logging, os, sys, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import planetary_computer as pc
import pystac_client
import stackstac
import rasterio
from rasterio.transform import from_origin
from scipy.ndimage import binary_erosion

logging.getLogger("rasterio._env").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Paths: works when run from the repo root or from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RESULTS = os.path.join(ROOT, "results")
SAMPLE = os.path.join(ROOT, "data", "sample_input")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(SAMPLE, exist_ok=True)
print("Python", sys.version.split()[0], "| root:", ROOT)

## 1. Configuration

All study parameters live here. Nothing below is hard-coded elsewhere.

- Sites were chosen from mangrove extent numbers (Global Mangrove Watch portal and ESA WorldCover class 95 counts per candidate box).
- Resolution: Abu Dhabi is processed at 20 m to keep memory manageable. Nabq is processed at 20 m (comparable reference) and at 10 m (detail).
- `OFFSET_DATE`: Sentinel-2 L2A products processed with baseline 04.00 or later (from 25 Jan 2022) carry a +1000 DN radiometric offset. We subtract it per scene before computing reflectance. The sanity check in section 3 confirms the two years end up on the same scale.

In [ ]:
CAT_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

SITES = {
    "abu_dhabi":      {"bbox": [54.40, 24.40, 54.60, 24.60], "epsg": 32640, "res": 20},
    "nabq_egypt_20m": {"bbox": [34.41, 28.16, 34.455, 28.21], "epsg": 32636, "res": 20},   # reference site, same grid size as Abu Dhabi
    "nabq_egypt_10m": {"bbox": [34.41, 28.16, 34.455, 28.21], "epsg": 32636, "res": 10},   # detail view
}
WINDOWS = {"baseline": "2019-01-01/2019-03-31", "recent": "2025-01-01/2025-03-31"}
CLOUD_MAX = 20                       # max scene cloud cover (%)
SCL_OK = [4, 5, 6]                   # SCL classes kept: vegetation, bare soil, water
OFFSET_DATE = np.datetime64("2022-01-25")

# Carbon densities in t C per ha (= Mg C/ha), from published arid-zone studies (see README)
CARBON = {
    "low  (Red Sea soil only, 1 m)":      43.0,                       # Almahasheer et al. 2017
    "mid  (Qatar tree + soil 1 m)":       10.18 + 50.17,              # Qatar A. marina stands
    "high (Qatar mid + 1 SD)":            10.18 + 0.82 + 50.17 + 6.27,
}
cat = pystac_client.Client.open(CAT_URL, modifier=pc.sign_inplace)
print({k: v["res"] for k, v in SITES.items()})

## 2. Functions

In [ ]:
def ndvi(r):
    return (r.sel(band="B08") - r.sel(band="B04")) / (r.sel(band="B08") + r.sel(band="B04"))

def ndre(r):
    return (r.sel(band="B08") - r.sel(band="B05")) / (r.sel(band="B08") + r.sel(band="B05"))

def composite(cat, bbox, epsg, res, dt):
    # Median surface-reflectance composite (B04, B05, B08) with cloud/shadow screening and offset correction.
    items = cat.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=dt,
                       query={"eo:cloud_cover": {"lt": CLOUD_MAX}}).item_collection()
    a = stackstac.stack(items, assets=["B04", "B05", "B08", "SCL"], bounds_latlon=bbox,
                        resolution=res, epsg=epsg, dtype="float64", fill_value=np.nan)
    ok = a.sel(band="SCL").isin(SCL_OK)
    offset = xr.where(a["time"] >= OFFSET_DATE, 1000.0, 0.0)
    refl = (a.sel(band=["B04", "B05", "B08"]) - offset) / 10000.0
    med = refl.where(ok).median("time").compute()
    return med, len(items)

def mangrove_mask(cat, bbox, epsg, res):
    # ESA WorldCover 2021, class 95 = mangroves, on the same grid as the composites.
    items = cat.search(collections=["esa-worldcover"], bbox=bbox, datetime="2021").item_collection()
    wc = stackstac.stack(items, assets=["map"], bounds_latlon=bbox, resolution=res, epsg=epsg,
                         dtype="float64", fill_value=np.nan).isel(time=0, band=0).compute()
    return wc == 95

def mad_sigma(v):
    return 1.4826 * np.median(np.abs(v - np.median(v)))

def summarize(name, res, r0, r1, mg):
    # Change statistics inside the full mangrove mask and inside the eroded core mask.
    core = mg.copy(data=binary_erosion(mg.values))
    px_km2 = res * res / 1e6
    d = ndvi(r1) - ndvi(r0)
    rows = []
    for mname, m in [("full_mask", mg), ("core_mask", core)]:
        v = d.where(m).values
        v = v[~np.isnan(v)]
        if v.size == 0:
            continue
        rows.append({
            "site": name, "mask": mname, "res_m": res,
            "mask_area_km2": round(float(m.sum()) * px_km2, 3),
            "valid_px": int(v.size),
            "median_ndvi_baseline": round(float(ndvi(r0).where(m).median()), 3),
            "median_ndvi_recent": round(float(ndvi(r1).where(m).median()), 3),
            "median_ndre_baseline": round(float(ndre(r0).where(m).median()), 3),
            "median_ndre_recent": round(float(ndre(r1).where(m).median()), 3),
            "pct_declined_lt_-0.05": round(100 * float((v < -0.05).mean()), 1),
            "pct_declined_lt_-0.10": round(100 * float((v < -0.10).mean()), 1),
            "pct_declined_lt_-0.20": round(100 * float((v < -0.20).mean()), 1),
            "pct_increased_gt_+0.10": round(100 * float((v > 0.10).mean()), 1),
            "change_median": round(float(np.median(v)), 3),
            "change_mad_sigma": round(float(mad_sigma(v)), 3),
        })
    return rows, d, core

def save_tif(da, path, epsg):
    x, y = da["x"].values, da["y"].values
    res = abs(float(x[1] - x[0]))
    tf = from_origin(float(x[0]) - res / 2, float(y[0]) + res / 2, res, res)
    arr = np.asarray(da.values, dtype="float32")
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype="float32", crs=f"EPSG:{epsg}", transform=tf, nodata=np.nan) as dst:
        dst.write(arr, 1)

## 3. Run all sites

Minutes of runtime, depending on the network. Each site loads two seasonal composites and the mangrove mask.
The printed **B08 quantiles** are the radiometric sanity check: both years must be on the same 0-1 reflectance scale (similar quantiles, no 1000-level offset).

In [ ]:
RES, STORE, ROWS = {}, {}, []
for name, cfg in SITES.items():
    r0, n0 = composite(cat, cfg["bbox"], cfg["epsg"], cfg["res"], WINDOWS["baseline"])
    r1, n1 = composite(cat, cfg["bbox"], cfg["epsg"], cfg["res"], WINDOWS["recent"])
    mg = mangrove_mask(cat, cfg["bbox"], cfg["epsg"], cfg["res"])
    q0 = r0.sel(band="B08").quantile([.05, .5, .95]).values.round(3)
    q1 = r1.sel(band="B08").quantile([.05, .5, .95]).values.round(3)
    print(f"{name}: scenes {n0} (2019) / {n1} (2025) | B08 q05/50/95 2019: {q0} 2025: {q1}")
    assert 0.0 < float(q0[1]) < 0.7 and 0.0 < float(q1[1]) < 0.7, "B08 median outside 0-0.7: check offset correction"
    rows, d, core = summarize(name, cfg["res"], r0, r1, mg)
    ROWS += rows
    STORE[name] = {"d": d, "mg": mg, "core": core, "r0": r0, "r1": r1, "n": (n0, n1)}

summary = pd.DataFrame(ROWS)
summary.to_csv(os.path.join(RESULTS, "summary.csv"), index=False)
summary.T

## 4. Reference-calibrated decline threshold

Nabq is a small, protected-looking and visually stable stand (to be confirmed with local sources). We use its **natural variability** (robust sigma of the NDVI change, 20 m grid, core mask) to set a decline threshold: median change minus three robust sigmas. This replaces an arbitrary cut-off. One reference site and two dates is weak evidence; it is stated as a limitation.

In [ ]:
ref = STORE["nabq_egypt_20m"]["d"].where(STORE["nabq_egypt_20m"]["core"]).values
ref = ref[~np.isnan(ref)]
THR = float(np.median(ref) - 3 * mad_sigma(ref))
print(f"Reference change: median {np.median(ref):.3f}, MAD-sigma {mad_sigma(ref):.3f} -> calibrated decline threshold {THR:.3f}")

cal = []
for name in ["abu_dhabi", "nabq_egypt_20m", "nabq_egypt_10m"]:
    v = STORE[name]["d"].where(STORE[name]["core"]).values
    v = v[~np.isnan(v)]
    cal.append({"site": name, "valid_core_px": int(v.size), "pct_below_calibrated_threshold": round(100 * float((v < THR).mean()), 1)})
calib = pd.DataFrame(cal)
calib.to_csv(os.path.join(RESULTS, "calibrated_threshold.csv"), index=False)
calib

## 5. Carbon stock range and exposed carbon

- **Stock** = full mangrove mask area x carbon density (low / mid / high).
- **Exposed** = area of core-mask pixels below the decline threshold, evaluated at -0.05, the calibrated threshold and -0.20, x density. It marks carbon sitting under vegetation that shows a decline signal. **It is not carbon that was lost.**
- Densities are from other sites (Qatar, Red Sea) and were not measured here. The low value covers soil only.

In [ ]:
out = []
for name in ["abu_dhabi", "nabq_egypt_10m"]:
    mg, core, d = STORE[name]["mg"], STORE[name]["core"], STORE[name]["d"]
    px_ha = SITES[name]["res"] ** 2 / 1e4
    stock_ha = float(mg.sum()) * px_ha
    core_ha = float(core.sum()) * px_ha
    v = d.where(core).values
    v = v[~np.isnan(v)]
    shares = [float((v < t).mean()) for t in (-0.05, THR, -0.20)]
    for label, dens in CARBON.items():
        out.append({"site": name, "density_case": label, "mangrove_ha": round(stock_ha, 1),
                    "stock_t_C": round(stock_ha * dens),
                    "exposed_t_C_min": round(core_ha * min(shares) * dens),
                    "exposed_t_C_max": round(core_ha * max(shares) * dens)})
carbon = pd.DataFrame(out)
carbon.to_csv(os.path.join(RESULTS, "carbon_range.csv"), index=False)
carbon

## 6. Maps and example outputs

Blue = NDVI increase, red = NDVI decrease, inside the core mangrove mask only.

In [ ]:
for name in ["abu_dhabi", "nabq_egypt_10m"]:
    d, core = STORE[name]["d"], STORE[name]["core"]
    fig, ax = plt.subplots(figsize=(7, 7))
    d.where(core).plot(ax=ax, vmin=-0.3, vmax=0.3, cmap="RdBu", cbar_kwargs={"label": "NDVI change 2025 - 2019"})
    ax.set_title(f"{name}: NDVI change inside mangrove core mask (Jan-Mar composites)")
    fig.savefig(os.path.join(RESULTS, f"{name}_ndvi_change.png"), dpi=130, bbox_inches="tight")
    plt.show()

## 7. Save a small example input

A small clipped sample for reviewers (Nabq, 10 m): the WorldCover mangrove mask and the two NDVI composites, plus the exact parameters used.

In [ ]:
s = STORE["nabq_egypt_10m"]
epsg = SITES["nabq_egypt_10m"]["epsg"]
save_tif(s["mg"].astype("float32"), os.path.join(SAMPLE, "nabq_worldcover_mangrove_mask_10m.tif"), epsg)
save_tif(ndvi(s["r0"]), os.path.join(SAMPLE, "nabq_ndvi_2019_JanMar_10m.tif"), epsg)
save_tif(ndvi(s["r1"]), os.path.join(SAMPLE, "nabq_ndvi_2025_JanMar_10m.tif"), epsg)
params = {"site": "nabq_egypt_10m", **SITES["nabq_egypt_10m"], "windows": WINDOWS, "cloud_max": CLOUD_MAX,
          "scl_ok": SCL_OK, "offset_date": str(OFFSET_DATE), "collections": ["sentinel-2-l2a", "esa-worldcover (2021)"],
          "scenes_used": s["n"], "calibrated_threshold": THR}
with open(os.path.join(SAMPLE, "params.json"), "w") as f:
    json.dump(params, f, indent=2)
print(sorted(os.listdir(SAMPLE)))

## 8. Record the exact package versions that ran

Copy the printed lines into `requirements.txt` so reviewers reproduce the same environment.

In [ ]:
from importlib.metadata import version
pk = ["pystac-client", "planetary-computer", "stackstac", "rasterio", "scipy", "numpy", "pandas", "xarray", "matplotlib", "dask"]
lock = [f"{p}=={version(p)}" for p in pk]
with open(os.path.join(ROOT, "requirements.lock.txt"), "w") as f:
    f.write("# Python " + sys.version.split()[0] + "\n" + "\n".join(lock) + "\n")
print("Python", sys.version.split()[0]); print(*lock, sep="\n")